# Lab 2 · Advanced RAG — reranking & evaluation (free Kaggle T4)

**Layer 3 · Data — Chapter 2 (the advanced pipeline).** Lab 1 built naive RAG. Here
we add the single biggest quality lever after chunking — a **cross-encoder
reranker** — and, crucially, **measure** whether it helps: **recall@3** and **MRR**,
vector-only vs reranked. Same **k8s Q&A dataset** and open models as Lab 1.

**Settings:** Accelerator = **GPU T4 x2** (not P100), Internet = **On**, then
**Run All**. The embedder + reranker run on CPU; only the final answer uses the GPU.

*Data: [`ItshMoh/kubernetes_qa_pairs`](https://huggingface.co/datasets/ItshMoh/kubernetes_qa_pairs).*

In [ ]:
import warnings, logging, os
warnings.filterwarnings('ignore')                 # silence library UserWarnings / deprecations
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'     # quiet model-load reports + generation-flag notes
os.environ['HF_HUB_VERBOSITY'] = 'error'           # quiet HF Hub 'unauthenticated requests' notice
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
for _n in ('huggingface_hub', 'sentence_transformers', 'transformers', 'datasets'):
    logging.getLogger(_n).setLevel(logging.ERROR)

!pip install -q faiss-cpu sentence-transformers datasets 2>/dev/null

import faiss, textwrap
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

## 1 · Rebuild the index (recap of Lab 1)

Same pipeline as Lab 1, condensed into one cell: pull the dataset, embed every
answer with **`bge-small`** (CPU), and load a flat **FAISS** index. The query uses
the same embedder. (Self-contained — no dependency on Lab 1's files.)

In [ ]:
from datasets import load_dataset
ds = load_dataset('ItshMoh/kubernetes_qa_pairs', split='train')

EMB_ID = 'BAAI/bge-small-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
embedder = SentenceTransformer(EMB_ID, device='cpu')

chunks = [{'id': f'qa-{i}', 'text': r['answer'], 'question': r['question'], 'topic': r['topic']}
          for i, r in enumerate(ds) if (r['answer'] or '').strip()]
vecs = embedder.encode([c['text'] for c in chunks], normalize_embeddings=True,
                       convert_to_numpy=True, batch_size=128).astype('float32')
index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

def retrieve(query, k=3):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = index.search(qv, k)
    return [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0])]

print('indexed', index.ntotal, 'answer-chunks')

## 2 · Reranker — put the best chunk first

Vector search is fast but blunt: it compares pre-computed vectors. A **cross-encoder
reranker** reads the query and each chunk *together* and re-scores them — much more
accurate. The pattern: **over-fetch** a generous top-k cheaply, then rerank and keep
the best few. Watch a chunk jump to #1.

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('BAAI/bge-reranker-base', device='cpu')

# Vector search alone grabs a loosely-related "images" chunk; the cross-encoder reads
# query+chunk together and promotes the real garbage-collection answer to the top.
q = 'How does Kubernetes clean up unused container images?'
cands = retrieve(q, k=10)                      # over-fetch cheaply
print('BEFORE rerank (vector order), top 3:')
for h in cands[:3]:
    print(f"  {h['score']:.3f}  {h['topic']}: {h['text'][:70]}...")

scores = reranker.predict([[q, h['text']] for h in cands])
for h, s in zip(cands, scores):
    h['rr'] = float(s)
reranked = sorted(cands, key=lambda h: h['rr'], reverse=True)
print('\nAFTER rerank (cross-encoder), top 3:')
for h in reranked[:3]:
    print(f"  rr={h['rr']:.2f}  {h['topic']}: {h['text'][:70]}...")

## 3 · Feed the reranked chunks to the model -> grounded, cited answer

Over-fetch a generous top-k, **rerank**, keep the best few, and let **Qwen2.5-3B**
answer *only* from them, with a citation. (This cell loads the LLM — ~6 GB on the
T4.) We also show the **refusal** when the context can't answer.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto').eval()
print('LLM on', model.device)

SYS = ('You are a Kubernetes assistant. Use ONLY the CONTEXT to answer, and cite the chunk ids in [brackets]. '
       "Do not use any outside knowledge. If the answer is not in the CONTEXT, reply exactly: I don't know.")

def answer(query, k=10, keep=3):
    cands = retrieve(query, k=k)                 # over-fetch
    scores = reranker.predict([[query, h['text']] for h in cands])
    for h, s in zip(cands, scores): h['rr'] = float(s)
    hits = sorted(cands, key=lambda h: h['rr'], reverse=True)[:keep]   # keep the best few
    ctx = '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
    text = tok.apply_chat_template(
        [{'role': 'system', 'content': SYS},
         {'role': 'user', 'content': f'CONTEXT:\n{ctx}\n\nQUESTION: {query}'}],
        tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=200, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip(), hits

ans, hits = answer('What is Role-Based Access Control (RBAC) in Kubernetes?')
print('loaded chunks:', [h['id'] for h in hits])
print('\nANSWER:\n', textwrap.fill(ans, 90))

In [ ]:
# refusal: nothing in a k8s corpus answers this
ans, _ = answer('What is the capital of France?')
print(textwrap.fill(ans, 90))

## 4 · Does it actually help? — evaluate retrieval

"The demo looked good" is not evidence. RAG has two stages that can fail; here we
measure the **retrieval** stage. Each question's own answer is its gold chunk, so we
can ask: does retrieval surface that chunk, and how high does it rank?

- **recall@3** — is the gold chunk in the top 3?
- **MRR** — 1 / rank of the gold chunk (rewards putting it first)

We score **vector-only** vs **+reranker** over a sample of questions. Watch the
reranker lift **MRR** (it puts the right chunk first) more than recall.

In [ ]:
EVAL_N = 50          # raise for a tighter estimate (slower: reranks EVAL_N x 10 pairs on CPU)
sample = chunks[:EVAL_N]

def rank_of(gold_id, ordered):
    for r, h in enumerate(ordered, 1):
        if h['id'] == gold_id:
            return r
    return None

def evaluate(use_rerank):
    rr_sum = hit3 = 0
    for c in sample:
        cands = retrieve(c['question'], k=10)             # over-fetch candidates
        if use_rerank:
            scores = reranker.predict([[c['question'], h['text']] for h in cands])
            for h, s in zip(cands, scores): h['rr'] = float(s)
            cands = sorted(cands, key=lambda h: h['rr'], reverse=True)
        r = rank_of(c['id'], cands)                       # rank of the gold chunk
        if r:
            rr_sum += 1 / r
            hit3 += (r <= 3)
    n = len(sample)
    return hit3 / n, rr_sum / n

rb, mb = evaluate(False)
rr, mr = evaluate(True)
print(f'over {EVAL_N} questions       recall@3    MRR')
print(f'  vector-only            {rb:5.2f}      {mb:5.2f}')
print(f'  + cross-encoder rerank {rr:5.2f}      {mr:5.2f}')

## Takeaways

- **Reranker = the biggest quality lever after chunking.** Over-fetch cheaply with
  vector search, then a cross-encoder re-reads query+chunk *together* and re-scores.
- It typically lifts **MRR** (ranking) more than **recall** — it doesn't find *more*
  right chunks, it puts the right one *first*. That directly improves the answer,
  since the model leans on the top chunks.
- **Always measure.** Evaluate retrieval (recall@k, MRR) separately from generation —
  it tells you *where* a failure is, not just *that* there is one.

Next levers (Chapter 2): **data tagging** for metadata filters, **query rewriting**
for intent, and **hybrid** (keyword + vector) search.